# 01 · Generador de landing
Simula la llegada incremental de datos desde sistemas origen.
Cada ejecución escribe un batch de archivos JSON en `landing.raw` con problemas
típicos de calidad (duplicados, nulos, precios como texto) para que el pipeline
tenga algo real que limpiar.

In [0]:
# Celda 1: parámetros
from pyspark.sql import functions as F

dbutils.widgets.text("batch_id", "1")
batch_id = int(dbutils.widgets.get("batch_id"))

LANDING = "/Volumes/workspace/landing/raw"
print(f"Generando batch {batch_id}")

In [0]:
# Celda 2: transacciones "sucias"
tx = (spark.table("samples.bakehouse.sales_transactions")
        .withColumn("bucket", F.abs(F.hash("transactionID")) % 5)
        .filter(F.col("bucket") == (batch_id - 1) % 5)
        .drop("bucket"))

dirty_tx = (tx
    .withColumn("unitPrice",
        F.when(F.rand() < 0.1,
               F.regexp_replace(F.col("unitPrice").cast("string"), r"\.", ","))
         .otherwise(F.col("unitPrice").cast("string")))
    .withColumn("quantity", F.when(F.rand() < 0.03, None).otherwise(F.col("quantity")))
    .withColumn("dateTime", F.date_format("dateTime", "yyyy-MM-dd HH:mm:ss"))
    .drop("cardNumber"))   # nunca bajar datos sensibles a landing

dirty_tx = dirty_tx.unionByName(dirty_tx.sample(0.05))   # ~5% duplicados

(dirty_tx.coalesce(1).write.mode("overwrite")
    .json(f"{LANDING}/transactions/batch_{batch_id:03d}"))

In [0]:
# Celda 3: clientes con cambios (para SCD tipo 2 más adelante)
cust = (spark.table("samples.bakehouse.sales_customers")
          .withColumn("city",
              F.when(F.rand() < 0.05, F.concat(F.col("city"), F.lit(" Norte")))
               .otherwise(F.col("city")))
          .withColumn("updated_at", F.current_timestamp()))

(cust.coalesce(1).write.mode("overwrite")
    .json(f"{LANDING}/customers/batch_{batch_id:03d}"))

In [0]:
# Celda 4: verificación
display(dbutils.fs.ls(f"{LANDING}/transactions"))
print("Filas transacciones:", dirty_tx.count())